In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set()
import warnings
warnings.filterwarnings('ignore')
import datetime
import json
import re
import gensim
from gensim import corpora, models
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.stem.porter import *
import nltk
# nltk.download('wordnet')
import random
import multiprocessing
from collections import defaultdict
from gensim.models import Word2Vec
from nltk.corpus import stopwords
stopwords = set(stopwords.words('english'))

In [2]:
base_path = '/home/luxiaoling/wukun/MIND/data/'

In [3]:
columns = ['uid', 'impr_time', 'week', 'hour', 'relative_days', 'relative_hours', 'relative_seconds', 'history', 'cur_impr']
train_df = pd.read_csv(base_path+'behavior_sample_train.tsv', sep='\t', names=columns)
train_df.head()


,uid,impr_time,week,hour,relative_days,relative_hours,relative_seconds,history,cur_impr
0,U495159,2019-11-09 00:00:13,5,0,6,144,518387,N88765 N74317 N63723 N121794 N95711 N64341 N75...,N104437-0 N76810-0 N98657-0 N25492-0 N108809-0...
1,U251164,2019-11-09 00:00:24,5,0,6,144,518376,N42703 N126834 N118005 N13066 N43872 N48258 N1...,N79990-0 N59303-0 N65783-0 N18956-0 N2198-0 N9...
2,U515173,2019-11-09 00:00:25,5,0,6,144,518375,N10243 N14424 N55661 N114369 N70847 N123633 N8...,N4789-0 N17547-0 N104610-0 N37049-0 N92463-0 N...
3,U528649,2019-11-09 00:00:37,5,0,6,144,518363,N116996 N31266 N71082 N113593 N107137 N49262 N...,N2198-0 N79990-0 N18956-0 N119284-0 N92613-0 N...
4,U701987,2019-11-09 00:00:41,5,0,6,144,518359,N119674 N128483 N74723 N64141 N25892,N17456-0 N62003-0 N129463-0 N25492-0 N18956-0 ...


In [4]:
columns = ['uid', 'impr_time', 'week', 'hour', 'relative_days', 'relative_hours', 'relative_seconds', 'history', 'cur_impr']
val_df = pd.read_csv(base_path+'behavior_sample_val.tsv', sep='\t', names=columns)
val_df.head()

,uid,impr_time,week,hour,relative_days,relative_hours,relative_seconds,history,cur_impr
0,U129523,2019-11-15 00:00:10,4,0,0,-23,-86390,N65386 N100173 N3472 N4858 N16620 N125867 N132...,N82503-0 N80770-0 N32993-0 N62800-0 N83707-1 N...
1,U411819,2019-11-15 00:00:19,4,0,0,-23,-86381,N102152 N26417 N60132 N79285 N75220 N108312 N8...,N100425-0 N40742-0 N83707-0 N73303-0 N26122-0 ...
2,U450360,2019-11-15 00:00:27,4,0,0,-23,-86373,N81071 N126696 N87146 N84689 N59870,N26122-0 N78508-0 N68624-0 N82503-0 N118623-1 ...
3,U423393,2019-11-15 00:00:32,4,0,0,-23,-86368,N1396 N44611 N42796 N86153 N70947 N15206 N1064...,N56139-0 N96473-0 N45410-0 N121136-0 N63322-0 ...
4,U145343,2019-11-15 00:00:39,4,0,0,-23,-86361,N6306 N54797 N41484 N91923 N75819 N85123 N1764...,N14857-0 N104054-0 N110967-0 N121411-0 N121375...


In [5]:
with open(base_path+'new_user.csv', 'r') as f:
    new_user = f.readline().split()

In [7]:
with open(base_path+'new_news.csv', 'r') as f:
    new_news = f.readline().split()

In [11]:
all_users = list(set(train_df.uid) | set(val_df.uid))
user2idx = dict(zip(all_users, range(len(all_users))))

In [15]:
uid_news_dict = defaultdict(set)
all_news = set()
for uid, history, cur_impr in zip(train_df.uid, train_df.history, train_df.cur_impr):
    try:
        history_news = history.split()
    except:
        history_news = []
    all_news.update(history_news)
    uid_news_dict[uid].update(history_news)
    imprs = cur_impr.split()
    pos_news = list(filter(lambda x: x[-1] == '1', imprs))
    pos_news = [x[:-2] for x in pos_news]
    impr_news = [x[:-2] for x in imprs]
    all_news.update(impr_news)
    uid_news_dict[uid].update(pos_news)

In [16]:
for uid, history, cur_impr in zip(val_df.uid, val_df.history, val_df.cur_impr):
    try:
        history_news = history.split()
    except:
        history_news = []
    all_news.update(history_news)
    uid_news_dict[uid].update(history_news)
    imprs = cur_impr.split()
    pos_news = list(filter(lambda x: x[-1] == '1', imprs))
    pos_news = [x[:-2] for x in pos_news]
    impr_news = [x[:-2] for x in imprs]
    all_news.update(impr_news)
    uid_news_dict[uid].update(pos_news)

In [17]:
len(all_news)

78052

In [18]:
news2idx = dict(zip(all_news, range(len(all_news))))

In [19]:
news2idx

{'N96836': 0,
 'N92414': 1,
 'N9099': 2,
 'N91931': 3,
 'N112375': 4,
 'N26956': 5,
 'N101537': 6,
 'N85177': 7,
 'N38461': 8,
 'N4534': 9,
 'N48993': 10,
 'N128786': 11,
 'N44381': 12,
 'N46673': 13,
 'N469': 14,
 'N76198': 15,
 'N2591': 16,
 'N37921': 17,
 'N63603': 18,
 'N49001': 19,
 'N60860': 20,
 'N125348': 21,
 'N36304': 22,
 'N77561': 23,
 'N116939': 24,
 'N97898': 25,
 'N29911': 26,
 'N3649': 27,
 'N126811': 28,
 'N10656': 29,
 'N3938': 30,
 'N27101': 31,
 'N13844': 32,
 'N80983': 33,
 'N122005': 34,
 'N71074': 35,
 'N30828': 36,
 'N28655': 37,
 'N49657': 38,
 'N32861': 39,
 'N101773': 40,
 'N27047': 41,
 'N119850': 42,
 'N67475': 43,
 'N118880': 44,
 'N107367': 45,
 'N45875': 46,
 'N102657': 47,
 'N110821': 48,
 'N87115': 49,
 'N44473': 50,
 'N43873': 51,
 'N35607': 52,
 'N109195': 53,
 'N58780': 54,
 'N22317': 55,
 'N52278': 56,
 'N7551': 57,
 'N73750': 58,
 'N1761': 59,
 'N101609': 60,
 'N40062': 61,
 'N120282': 62,
 'N92808': 63,
 'N40512': 64,
 'N21309': 65,
 'N56879': 66

In [26]:
save_path = '/home/luxiaoling/wukun/pinsage/data/'

In [27]:
with open(save_path+'user2idx.json', 'w') as f:
    json.dump(user2idx, f, indent=4)

In [28]:
with open(save_path+'news2idx.json', 'w') as f:
    json.dump(news2idx, f, indent=4)

In [29]:
user2news = {k: '_'.join(v) for k, v in uid_news_dict.items()}

In [30]:
with open(save_path+'user2news.json', 'w') as f:
    json.dump(user2news, f, indent=4)